# 22 — Information Extraction, Extractive QA & Summarization

**Learning objective.** Combine entity/rule extraction with TF-IDF sentence retrieval to build transparent applied NLP baselines.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**document + task → evidence selection/extraction/generation → task output**

Follow the information transformation first; treat the API as an implementation detail.

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Change **task boundary** from extractive → generative | output freedom increases | fluency rises but unsupported content risk appears |
| Change retrieval/scoring representation | selected evidence changes | answer/summary can change even with same generator |
| Tighten evidence policy | fewer unsupported outputs pass | coverage may fall while trust improves |

> Write down what should move downstream before changing a control.

## Think before running the next cell

1. Why is extractive QA easier to audit than free generation?
2. If the wrong sentence is retrieved, can a perfect extractor still answer correctly?

### When to use
Use extraction when the answer/evidence already exists in text; use generation when transformation/composition is genuinely needed.

### When not to use / caution
Do not use free generation where exact provenance or regulatory traceability is mandatory without evidence controls.

### Debugging lens
Debug in stages: evidence selection first, then extraction/generation—not the final text alone.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
text='''NielsenIQ opened an AI research program in Chennai. The team builds machine-learning systems for retail analytics. The program focuses on scalable experimentation and reliable deployment.'''
sentences=[s.strip() for s in re.split(r'(?<=[.!?])\s+',text) if s.strip()]
print(pd.DataFrame({'sentence_id':range(len(sentences)),'sentence':sentences}).to_string(index=False))

 sentence_id                                                                 sentence
           0                      NielsenIQ opened an AI research program in Chennai.
           1           The team builds machine-learning systems for retail analytics.
           2 The program focuses on scalable experimentation and reliable deployment.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
question='Where is the AI research program?'
vec=TfidfVectorizer(stop_words='english')
M=vec.fit_transform(sentences+[question])
scores=cosine_similarity(M[-1],M[:-1]).ravel()
best=int(scores.argmax())
print('Question:',question)
print('Retrieved answer sentence:',sentences[best])
print('score:',round(float(scores[best]),3))

Question: Where is the AI research program?
Retrieved answer sentence: NielsenIQ opened an AI research program in Chennai.
score: 0.596


In [4]:
# Extractive summary: choose sentence with highest average similarity to the rest.
S=cosine_similarity(M[:-1])
centrality=(S.sum(axis=1)-1)/(len(sentences)-1)
summary=sentences[int(centrality.argmax())]
print('One-sentence extractive summary:',summary)

One-sentence extractive summary: The program focuses on scalable experimentation and reliable deployment.


---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Separate extraction/retrieval from generation
- Build a transparent QA/summarization baseline